# Phase 2B - Utilization lab

**Phase 2B - The 3-arm utilization lab (independent of retrieval hit).** Independent notebook - runs standalone in Colab or locally (offline, no key).

## 0. Setup (self-contained)

In [ ]:
# Self-contained setup - works standalone in Google Colab or locally.
import sys, os, subprocess
from pathlib import Path
REPO_URL = "https://github.com/syaikhipin/kdd26-memdiag"
try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
if IN_COLAB:
    repo = Path("/content/kdd26-memdiag")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=False)
    SOURCE = repo / "experiment" / "github_submission" / "source"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "matplotlib", "pyyaml"], check=False)
else:
    SOURCE = None
    for cand in [Path.cwd(), *Path.cwd().parents]:
        for sub in ("source", "experiment"):
            if (cand / sub / "run.py").exists():
                SOURCE = cand / sub
                break
        if SOURCE:
            break
    if SOURCE is None:
        raise FileNotFoundError("Run from the repo root (or in Colab it auto-clones).")
sys.path.insert(0, str(SOURCE))
os.environ.setdefault("OPENAI_BASE_URL", "https://api.openai.com/v1")
SOURCE_DIR = SOURCE
PROJECT_ROOT = SOURCE.parent
RESULTS_DIR = PROJECT_ROOT / "results"
print("SOURCE_DIR =", SOURCE, "| IN_COLAB =", IN_COLAB)

print("SOURCE_DIR =", SOURCE_DIR)

`provider_rate - p0` is the honest memory gain; `oracle_rate` is the ceiling.

In [ ]:
import pathlib
cands = sorted(pathlib.Path(RESULTS_DIR).glob('run_*_real_utilization_lab.tsv')) \
 + sorted((RESULTS_DIR/'summaries').glob('*utilization_lab.tsv'))
lab = cands[-1] if cands else None
print(lab.name, ':\n'); print(lab.read_text()) if lab else print('Run --mode real first')

## Probe one failure in the raw trace

In [ ]:
import json
raws = sorted(pathlib.Path(RESULTS_DIR).glob('run_*_locomo_raw.json'))
raw = json.loads(raws[-1].read_text()) if raws else None
if raw:
 rec = next((r for r in raw['records'] if r.get('failure_category')=='retrieval_miss'), raw['records'][0])
 print('Q:', rec['question'][:70]); print('gold:', rec['evidence_ids'][:5]); print('retrieved:', rec['retrieved_memory_ids'][:5])
else: print('(no raw trace - run the diagnose notebook first)')